#  <font color='#FFE15D'><b>💎 Dataset Preparation for Instruction Tuning</b></font>

# 🔴 **Environment Setup**

## 🟠 Change the font size of the output cells

In [1]:
print('Salam Howsam!')

Salam Howsam!


In [2]:
from IPython.display import HTML
shell = get_ipython()

def adjust_font_size():
  display(HTML('''<style>
    body {
      font-size: 24px;
    }
  '''))

if adjust_font_size not in shell.events.callbacks['pre_execute']:
  shell.events.register('pre_execute', adjust_font_size)

In [3]:
print('Salam Howsam!')

Salam Howsam!


## 🟠 `pip`

In [ ]:
# !pip install -q datasets

# 🔴 **Import**

In [4]:
import json
from tqdm import tqdm
from pprint import pprint

from datasets import load_dataset
from tokenizers import Tokenizer

import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

# 🔴 **Load TinyStories Instruction Dataset**

In [5]:
dataset = load_dataset("roneneldan/TinyStoriesInstruct")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


TinyStories-Instruct-train.txt:   0%|          | 0.00/2.66G [00:00<?, ?B/s]

TinyStories-Instruct-valid.txt:   0%|          | 0.00/26.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/21755681 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/218380 [00:00<?, ? examples/s]

In [6]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 21755681
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 218380
    })
})

In [11]:
dataset['train'][:50]

{'text': ['Features: Dialogue',
  'Words: quit, oak, gloomy',
  'Summary: Sara and Ben were playing in the park, but Sara wanted to go home because it was cold and dark. Ben convinced her to stay and play, but eventually agreed to go home and have hot cocoa.',
  'Story: ',
  '',
  'Sara and Ben were playing in the park. They liked to climb the big oak tree and pretend they were birds. They made nests with leaves and twigs and sang songs.',
  'But today, the sky was gloomy and the wind was cold. Sara felt sad and cold. She wanted to go home and have some hot cocoa.',
  '"Ben, I want to quit," she said. "It\'s too cold and dark. Let\'s go home."',
  'Ben looked at Sara and frowned. He liked the oak tree and the park. He wanted to stay and play.',
  '"No, Sara, don\'t quit," he said. "It\'s fun here. Look, there\'s a squirrel. Let\'s chase it."',
  "Sara shook her head. She didn't want to chase the squirrel. She wanted to go home and have some hot cocoa.",
  '"Please, Ben, let\'s go home,

In [12]:
n = 5
num_samples = 0
for line in dataset['train']:
    pprint(">>> " + line['text'])
    if line['text'] == '<|endoftext|>':
        print("\n" + 50*"=" + "\n")
        num_samples += 1
        if num_samples == n:
            break

'>>> Features: Dialogue'
'>>> Words: quit, oak, gloomy'
('>>> Summary: Sara and Ben were playing in the park, but Sara wanted to go '
 'home because it was cold and dark. Ben convinced her to stay and play, but '
 'eventually agreed to go home and have hot cocoa.')
'>>> Story: '
'>>> '
('>>> Sara and Ben were playing in the park. They liked to climb the big oak '
 'tree and pretend they were birds. They made nests with leaves and twigs and '
 'sang songs.')
('>>> But today, the sky was gloomy and the wind was cold. Sara felt sad and '
 'cold. She wanted to go home and have some hot cocoa.')
('>>> "Ben, I want to quit," she said. "It\'s too cold and dark. Let\'s go '
 'home."')
('>>> Ben looked at Sara and frowned. He liked the oak tree and the park. He '
 'wanted to stay and play.')
('>>> "No, Sara, don\'t quit," he said. "It\'s fun here. Look, there\'s a '
 'squirrel. Let\'s chase it."')
(">>> Sara shook her head. She didn't want to chase the squirrel. She wanted "
 'to go home and ha

In [13]:
counter = 0
for line in tqdm(dataset["train"], desc="Counting Stories"):
    if line["text"].startswith("Story:"):
        counter += 1
print(f"\n{counter:_}")

Counting Stories: 100%|██████████| 21755681/21755681 [08:13<00:00, 44108.16it/s]


2_476_533


# 🔴 **Dataset Preparation**

## 🟠 Clean Instruction Dataset

In [15]:
def extract_stories(dataset, section, max_samples=None):
    """
    Extracts story samples from the dataset in a structured format.
    Assumes dataset[section] supports indexing (like HuggingFace Dataset).
    """
    num_rows = len(dataset[section])
    i = 0  # index into dataset[section]
    all_samples = []
    sample = {"summary": None, "features": None, "sentence": None, "words": None, "story": None}

    with tqdm(total=num_rows, desc=f"Processing {section}") as pbar:
        while i < num_rows:
            row = dataset[section][i]
            line = row["text"].strip()
            i += 1
            pbar.update(1)  # we've consumed one row

            if line.startswith("Summary:"):
                sample["summary"] = line.replace("Summary:", "").strip()

            elif line.startswith("Features:"):
                sample["features"] = line.replace("Features:", "").strip()

            elif line.startswith("Sentence:") or line.startswith("Random sentence:"):
                sample["sentence"] = line.replace("Sentence:", "").replace("Random sentence:", "").strip()

            elif line.startswith("Words:"):
                sample["words"] = line.replace("Words:", "").strip()

            elif line.startswith("Story:"):
                story_lines = []
                # consume following lines until end token or next Summary
                while i < num_rows:
                    next_row = dataset[section][i]
                    next_line = next_row["text"].strip()
                    i += 1
                    pbar.update(1)  # count this consumed row too

                    if next_line == "<|endoftext|>":
                        break
                    if next_line.startswith("Summary:"):
                        sample["summary"] = next_line.replace("Summary:", "").strip()
                        break

                    story_lines.append(next_line)

                sample["story"] = "\n\n".join(story_lines).strip("\n")
                all_samples.append(sample)
                sample = {"summary": None, "features": None, "sentence": None, "words": None, "story": None}

                if max_samples and len(all_samples) >= max_samples:
                    break

    return all_samples

In [16]:
section = "train"

# 1. Extract samples
samples = extract_stories(dataset, section=section, max_samples=10)

Processing train:   0%|          | 158/21755681 [00:00<42:03, 8620.60it/s]


In [18]:
samples[0]

{'summary': 'Sara and Ben were playing in the park, but Sara wanted to go home because it was cold and dark. Ben convinced her to stay and play, but eventually agreed to go home and have hot cocoa.',
 'features': 'Dialogue',
 'sentence': None,
 'words': 'quit, oak, gloomy',
 'story': 'Sara and Ben were playing in the park. They liked to climb the big oak tree and pretend they were birds. They made nests with leaves and twigs and sang songs.\n\nBut today, the sky was gloomy and the wind was cold. Sara felt sad and cold. She wanted to go home and have some hot cocoa.\n\n"Ben, I want to quit," she said. "It\'s too cold and dark. Let\'s go home."\n\nBen looked at Sara and frowned. He liked the oak tree and the park. He wanted to stay and play.\n\n"No, Sara, don\'t quit," he said. "It\'s fun here. Look, there\'s a squirrel. Let\'s chase it."\n\nSara shook her head. She didn\'t want to chase the squirrel. She wanted to go home and have some hot cocoa.\n\n"Please, Ben, let\'s go home," she sa

In [20]:
def save_stories_to_jsonl(samples, output_path):
    """
    Saves extracted story samples into a JSONL file after validating completeness.

    Args:
        samples (list[dict]): Output list from `extract_stories()`.
        output_path (str): File path to save the JSONL file.
    """

    valid_samples = []

    # Validate samples before saving
    for sample in tqdm(samples, desc="🧩 Validating samples"):
        # Check that 'story' exists and at least one of the others is not empty
        if not sample.get("story"):
            continue
        if not any(sample.get(k) for k in ["summary", "features", "sentence", "words"]):
            continue

        # Optional cleanup: remove empty fields (if you prefer cleaner files)
        clean_sample = {k: v for k, v in sample.items() if v}
        valid_samples.append(clean_sample)

    # Save to JSONL
    with open(output_path, "w", encoding="utf-8") as f_out:
        for s in valid_samples:
            f_out.write(json.dumps(s, ensure_ascii=False) + "\n")

    print(f"✅ Done. Saved {len(valid_samples):,} valid samples to: {output_path}")

    return len(valid_samples)

In [22]:
section = "train"

# 1. Extract samples
samples = extract_stories(dataset, section=section, max_samples=1000)

# 2. Save them
output_path = f"dataset/processed/tinystories_instruct_cleaned_{section}_data.jsonl"
num_train_samples = save_stories_to_jsonl(samples, output_path)

🧩 Validating samples: 100%|██████████| 1000/1000 [00:00<00:00, 427379.66it/s]

✅ Done. Saved 985 valid samples to: dataset/processed/tinystories_instruct_cleaned_train_data.jsonl


In [23]:
section = "train"

# Path to the cleaned .jsonl file
jsonl_path = f"dataset/processed/tinystories_instruct_cleaned_{section}_data.jsonl"

# Number of samples to preview
num_samples_to_show = 10

print(f"🔍 Reading first {num_samples_to_show} samples from {jsonl_path}\n")

# Read line by line
with open(jsonl_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= num_samples_to_show:
            break

        sample = json.loads(line.strip())

        print(f"===== Sample {i+1}", 50*"=")
        print("📝 Summary:", sample.get("summary", None))
        print("⭐ Features:", sample.get("features", None))
        print("💬 Sentence:", sample.get("sentence", None))
        print("🧩 Words:", sample.get("words", None))
        pprint(f'📖 Story: \n{sample.get("story", None)}')
        print()

🔍 Reading first 10 samples from dataset/processed/tinystories_instruct_cleaned_train_data.jsonl

===== Sample 1 ==================================================
📝 Summary: Sara and Ben were playing in the park, but Sara wanted to go home because it was cold and dark. Ben convinced her to stay and play, but eventually agreed to go home and have hot cocoa.
⭐ Features: Dialogue
💬 Sentence: None
🧩 Words: quit, oak, gloomy
('📖 Story: \n'
 'Sara and Ben were playing in the park. They liked to climb the big oak tree '
 'and pretend they were birds. They made nests with leaves and twigs and sang '
 'songs.\n'
 '\n'
 'But today, the sky was gloomy and the wind was cold. Sara felt sad and cold. '
 'She wanted to go home and have some hot cocoa.\n'
 '\n'
 '"Ben, I want to quit," she said. "It\'s too cold and dark. Let\'s go home."\n'
 '\n'
 'Ben looked at Sara and frowned. He liked the oak tree and the park. He '
 'wanted to stay and play.\n'
 '\n'
 '"No, Sara, don\'t quit," he said. "It\'s fun

In [ ]:
section = "train"

# 1. Extract samples
samples = extract_stories(dataset, section=section)

# 2. Save them
output_path = f"dataset/processed/tinystories_instruct_cleaned_{section}_data.jsonl"
num_train_samples = save_stories_to_jsonl(samples, output_path)

In [ ]:
section = "validation"

# 1. Extract samples
samples = extract_stories(dataset, section=section)

# 2. Save them
output_path = f"dataset/processed/tinystories_instruct_cleaned_{section}_data.jsonl"
num_valid_samples = save_stories_to_jsonl(samples, output_path)

## 🟠 Convert Samples to Format {Prompt + Completion}

In [24]:
def build_instruction_finetune_data(section, input_path, output_path, num_samples=None):
    """
    Converts cleaned instruct dataset into prompt-completion pairs for fine-tuning.

    Args:
        section (str): Dataset split name (e.g. 'train', 'validation').
    """

    # Read input line-by-line
    with open(input_path, "r", encoding="utf-8") as f_in, open(output_path, "w", encoding="utf-8") as f_out:
        for i, line in enumerate(tqdm(f_in, total=num_samples, desc="🔧 Building prompts")):
            if i == num_samples:
                break
            item = json.loads(line.strip())

            # Build prompt
            prompt_parts = ["Give a short story."]

            if item.get("words"):
                prompt_parts.append(f"The story should include these words: {item['words']}.")

            if item.get("sentence"):
                prompt_parts.append(f"Use this sentence somewhere in the story: {item['sentence']}")

            if item.get("summary"):
                prompt_parts.append(f"The story is about: {item['summary']}.")

            prompt_parts.append("Now complete the story:")

            # Final prompt string
            prompt = "\n".join(prompt_parts)

            # Completion is just the story
            completion = item["story"].strip()

            # Write to JSONL
            f_out.write(json.dumps({
                "prompt": prompt,
                "completion": completion
            }, ensure_ascii=False) + "\n")

    print(f"✅ Done. Saved to {output_path}")

In [25]:
section = "train"

# Input file: cleaned instruct dataset
input_path = f"dataset/processed/tinystories_instruct_cleaned_{section}_data.jsonl"

# Output file: prompt + completion pairs
output_path = f"dataset/processed/instruction_finetune_{section}_data.jsonl"

build_instruction_finetune_data(section, input_path, output_path, num_train_samples)

🔧 Building prompts: 100%|██████████| 985/985 [00:00<00:00, 25462.64it/s]

✅ Done. Saved to dataset/processed/instruction_finetune_train_data.jsonl


In [ ]:
section = "validation"

# Input file: cleaned instruct dataset
input_path = f"dataset/processed/tinystories_instruct_cleaned_{section}_data.jsonl"

# Output file: prompt + completion pairs
output_path = f"dataset/processed/instruction_finetune_{section}_data.jsonl"

build_instruction_finetune_data(section, input_path, output_path, num_valid_samples)

In [26]:
section = "train"

# Path to the final instruction-tuned JSONL file
jsonl_path = f"dataset/processed/instruction_finetune_{section}_data.jsonl"

# Number of samples to preview
num_samples_to_show = 10

print(f"🔍 Reading first {num_samples_to_show} samples from {jsonl_path}\n")

# Read line by line
with open(jsonl_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= num_samples_to_show:
            break

        sample = json.loads(line.strip())

        print(f"===== Sample {i+1}", 60*"=")
        pprint(f'🧭 Prompt:\n{sample.get("prompt", "[No prompt]")}')
        pprint(f'\n📖 Completion:\n{sample.get("completion", "[No completion]")}')
        print()

🔍 Reading first 10 samples from dataset/processed/instruction_finetune_train_data.jsonl

===== Sample 1 ============================================================
('🧭 Prompt:\n'
 'Give a short story.\n'
 'The story should include these words: quit, oak, gloomy.\n'
 'The story is about: Sara and Ben were playing in the park, but Sara wanted '
 'to go home because it was cold and dark. Ben convinced her to stay and play, '
 'but eventually agreed to go home and have hot cocoa..\n'
 'Now complete the story:')
('\n'
 '📖 Completion:\n'
 'Sara and Ben were playing in the park. They liked to climb the big oak tree '
 'and pretend they were birds. They made nests with leaves and twigs and sang '
 'songs.\n'
 '\n'
 'But today, the sky was gloomy and the wind was cold. Sara felt sad and cold. '
 'She wanted to go home and have some hot cocoa.\n'
 '\n'
 '"Ben, I want to quit," she said. "It\'s too cold and dark. Let\'s go home."\n'
 '\n'
 'Ben looked at Sara and frowned. He liked the oak tree a

## 🟠 Convert to Tensors

In [27]:
def tokenize_instruction_data(section, input_path, tokenizer, output_path, num_samples=None):
    """
    Tokenizes prompt-completion pairs for a given dataset section.

    Args:
        section (str): Dataset split name ('train', 'validation', etc.).
        tokenizer: A tokenizer object with an `.encode()` method (e.g., from tiktoken or SentencePiece).
    """

    with open(input_path, "r", encoding="utf-8") as f_in, open(output_path, "w", encoding="utf-8") as f_out:
        for line in tqdm(f_in, total=num_samples, desc=f"Tokenizing {section} set"):
            data = json.loads(line)
            prompt = data["prompt"].strip()
            completion = data["completion"].strip()

            # Tokenize separately
            prompt_ids = tokenizer.encode(prompt).ids
            completion_ids = tokenizer.encode(completion).ids

            # Save as one JSONL line
            f_out.write(json.dumps({
                "prompt_ids": prompt_ids,
                "completion_ids": completion_ids,
            }) + "\n")

    print(f"✅ Saved tokenized {section} set to: {output_path}")

In [29]:
tokenizer = Tokenizer.from_file("weight/bpe-tokenizer_tinystories.json")

In [31]:
print(sample["prompt"])

Give a short story.
The story should include these words: crawl, card, furry.
Use this sentence somewhere in the story: They say "Thank you, mom and dad!
The story is about: Lily and Ben find a surprise box with a teddy bear, hat, scarf, ball, and a card on their birthday, and their parents later come with a real cake and candles to celebrate..
Now complete the story:


In [34]:
print(tokenizer.encode(sample["prompt"]).tokens)

['<|endoftext|>', 'Give', 'Ġa', 'Ġshort', 'Ġstory', '.', 'Ċ', 'The', 'Ġstory', 'Ġshould', 'Ġinclude', 'Ġthese', 'Ġwords', ':', 'Ġcrawl', ',', 'Ġcard', ',', 'Ġfurry', '.', 'Ċ', 'U', 'se', 'Ġthis', 'Ġsent', 'ence', 'Ġsomewhere', 'Ġin', 'Ġthe', 'Ġstory', ':', 'ĠThey', 'Ġsay', 'Ġ"', 'Thank', 'Ġyou', ',', 'Ġmom', 'Ġand', 'Ġdad', '!', 'Ċ', 'The', 'Ġstory', 'Ġis', 'Ġabout', ':', 'ĠLily', 'Ġand', 'ĠBen', 'Ġfind', 'Ġa', 'Ġsurprise', 'Ġbox', 'Ġwith', 'Ġa', 'Ġteddy', 'Ġbear', ',', 'Ġhat', ',', 'Ġscarf', ',', 'Ġball', ',', 'Ġand', 'Ġa', 'Ġcard', 'Ġon', 'Ġtheir', 'Ġbirthday', ',', 'Ġand', 'Ġtheir', 'Ġparents', 'Ġlater', 'Ġcome', 'Ġwith', 'Ġa', 'Ġreal', 'Ġcake', 'Ġand', 'Ġcandles', 'Ġto', 'Ġcelebrate', '..', 'Ċ', 'Now', 'Ġcomplete', 'Ġthe', 'Ġstory', ':']


In [35]:
print(tokenizer.encode(sample["prompt"]).ids)

[1, 3687, 155, 4152, 1129, 15, 132, 269, 1129, 818, 5904, 2463, 1665, 27, 3609, 13, 2194, 13, 2225, 15, 132, 54, 283, 636, 3829, 1492, 3760, 212, 159, 1129, 27, 216, 385, 225, 921, 242, 13, 261, 161, 544, 2, 132, 269, 1129, 303, 564, 27, 260, 161, 371, 534, 155, 1244, 679, 238, 155, 1535, 690, 13, 1159, 13, 2489, 13, 577, 13, 161, 155, 2194, 241, 349, 1970, 13, 161, 349, 1228, 2053, 772, 238, 155, 878, 1078, 161, 4558, 162, 3533, 4245, 132, 2371, 4170, 159, 1129, 27]


In [36]:
section = "train"

input_path = f"dataset/processed/instruction_finetune_{section}_data.jsonl"

output_path = f"dataset/processed/tokenized_instruction_{section}_data.jsonl"

tokenize_instruction_data(section, input_path, tokenizer, output_path, num_train_samples)

Tokenizing train set: 100%|██████████| 985/985 [00:00<00:00, 1189.83it/s]

✅ Saved tokenized train set to: dataset/processed/tokenized_instruction_train_data.jsonl


In [ ]:
section = "validation"

input_path = f"dataset/processed/instruction_finetune_{section}_data.jsonl"

output_path = f"dataset/processed/tokenized_instruction_{section}_data.jsonl"

tokenize_instruction_data(section, input_path, tokenizer, output_path, num_valid_samples)

In [40]:
# Path to the tokenized dataset
section = "train"
input_path = f"dataset/processed/tokenized_instruction_{section}_data.jsonl"

# Number of samples to preview
num_samples_to_show = 3

print(f"🔍 Reading first {num_samples_to_show} samples from {input_path}\n")

with open(input_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= num_samples_to_show:
            break

        sample = json.loads(line.strip())

        print(f"===== Sample {i+1}", 60*"=")
        print("🆔 prompt_ids:", sample.get("prompt_ids", [])[:20], "...")  # truncate for readability
        print("🏷️ completion_ids:", sample.get("completion_ids", [])[:20], "...")
        print()

🔍 Reading first 3 samples from dataset/processed/tokenized_instruction_train_data.jsonl

===== Sample 1 ============================================================
🆔 prompt_ids: [1, 3687, 155, 4152, 1129, 15, 132, 269, 1129, 818, 5904, 2463, 1665, 27, 4185, 13, 4610, 13, 3612, 15] ...
🏷️ completion_ids: [1, 1059, 161, 371, 325, 501, 212, 159, 465, 15, 216, 511, 162, 909, 159, 302, 4610, 575, 161, 1175] ...

===== Sample 2 ============================================================
🆔 prompt_ids: [1, 3687, 155, 4152, 1129, 15, 132, 269, 1129, 818, 5904, 2463, 1665, 27, 1515, 13, 993, 13, 1809, 15] ...
🏷️ completion_ids: [1, 239, 511, 162, 1515, 205, 1645, 15, 209, 3175, 200, 520, 252, 951, 993, 15, 417, 2465, 178, 155] ...

===== Sample 3 ============================================================
🆔 prompt_ids: [1, 3687, 155, 4152, 1129, 15, 132, 269, 1129, 818, 5904, 2463, 1665, 27, 4923, 13, 2539, 13, 1017, 15] ...
🏷️ completion_ids: [1, 814, 161, 581, 325, 2991, 488, 511, 162, 255,

## 🟠 Copy Files to Modal Volume

In [ ]:
# import os
# import shutil

# # Create folder
# os.makedirs("/mnt/llm-tinystories/data/instruction-tuning", exist_ok=True)

# # Copy files
# shutil.copy("tokenized_instruction_train_data.jsonl", "/mnt/llm-tinystories/data/instruction-tuning")
# shutil.copy("tokenized_instruction_validation_data.jsonl", "/mnt/llm-tinystories/data/instruction-tuning")

# print("✅ Files copied successfully!")

# 🔴 **Custom Dataset & Dataloader**

## 🟠 Custom Dataset

In [41]:
class InstructionDataset(Dataset):
    def __init__(self, path, max_samples=None, max_total_length=None):
        """
        path: path to tokenized .jsonl file
        max_samples: number of samples to load (None = load all)
        max_total_length: only keep samples where len(prompt_ids) + len(completion_ids) <= this value
        """
        i = 0
        self.samples = []
        with open(path, "r", encoding="utf-8") as f:
            for line in tqdm(f, total=max_samples, desc=f"📂 Loading {path}"):
                if max_samples and (i >= max_samples):
                    break

                sample = json.loads(line.strip())

                total_len = len(sample["prompt_ids"]) + len(sample["completion_ids"])
                if max_total_length is not None and total_len > max_total_length:
                    continue  # skip this sample

                i += 1
                self.samples.append(sample)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        return sample["prompt_ids"], sample["completion_ids"]

In [42]:
train_set = InstructionDataset(
    "dataset/processed/tokenized_instruction_train_data.jsonl",
    max_samples=10_000,
    max_total_length=512
)

# valid_set = InstructionDataset(
#     "dataset/processed/tokenized_instruction_validation_data.jsonl",
#     max_samples=num_valid_samples,
#     max_total_length=512
# )

📂 Loading dataset/processed/tokenized_instruction_train_data.jsonl:  10%|▉         | 985/10000 [00:00<00:00, 20379.38it/s]


In [43]:
len(train_set)

899

In [47]:
print(train_set[0][0])

[1, 3687, 155, 4152, 1129, 15, 132, 269, 1129, 818, 5904, 2463, 1665, 27, 4185, 13, 4610, 13, 3612, 15, 132, 269, 1129, 303, 564, 27, 649, 161, 371, 325, 501, 212, 159, 465, 13, 306, 649, 340, 162, 332, 483, 683, 200, 178, 1303, 161, 1269, 15, 371, 7912, 205, 162, 788, 161, 255, 13, 306, 2240, 1389, 162, 332, 483, 161, 358, 1342, 3731, 4245, 132, 2371, 4170, 159, 1129, 27]


In [48]:
pprint(tokenizer.decode(train_set[0][0]))

('Give a short story.\n'
 'The story should include these words: quit, oak, gloomy.\n'
 'The story is about: Sara and Ben were playing in the park, but Sara wanted '
 'to go home because it was cold and dark. Ben convinced her to stay and play, '
 'but eventually agreed to go home and have hot cocoa..\n'
 'Now complete the story:')


In [49]:
pprint(tokenizer.decode(train_set[0][1]))

('Sara and Ben were playing in the park. They liked to climb the big oak tree '
 'and pretend they were birds. They made nests with leaves and twigs and sang '
 'songs.\n'
 '\n'
 'But today, the sky was gloomy and the wind was cold. Sara felt sad and cold. '
 'She wanted to go home and have some hot cocoa.\n'
 '\n'
 '"Ben, I want to quit," she said. "It\'s too cold and dark. Let\'s go home."\n'
 '\n'
 'Ben looked at Sara and frowned. He liked the oak tree and the park. He '
 'wanted to stay and play.\n'
 '\n'
 '"No, Sara, don\'t quit," he said. "It\'s fun here. Look, there\'s a '
 'squirrel. Let\'s chase it."\n'
 '\n'
 "Sara shook her head. She didn't want to chase the squirrel. She wanted to go "
 'home and have some hot cocoa.\n'
 '\n'
 '"Please, Ben, let\'s go home," she said. "We can play here another day. I\'m '
 'cold and hungry."\n'
 '\n'
 'Ben saw that Sara was shivering and looked unhappy. He loved his sister and '
 "didn't want her to be sad. He nodded and smiled.\n"
 '\n'
 '

## 🟠 Dataloader

In [106]:
# def pad_collate(batch):
#     prompt_list = []
#     completion_list = []
#     for prompt_ids, completion_ids in batch:
#         prompt_ids = torch.tensor(prompt_ids, dtype=torch.long)
#         completion_ids = torch.tensor(completion_ids, dtype=torch.long)
#         prompt_list.append(prompt_ids)
#         completion_list.append(completion_ids)

#     prompt_padded = pad_sequence(prompt_list, batch_first=True, padding_value=1)
#     completion_padded = pad_sequence(completion_list, batch_first=True, padding_value=1)

#     return prompt_padded, completion_padded

In [134]:
def pad_collate(batch):
    input_ids_list = []
    target_ids_list = []

    for prompt_ids, completion_ids in batch:
        # Full ids
        full_ids = prompt_ids + completion_ids[1:] + [1]

        # Convert to tensor
        input_ids = torch.tensor(full_ids[:-1], dtype=torch.long)
        target_ids = torch.tensor(full_ids[1:], dtype=torch.long)

        target_ids[:len(prompt_ids)] = -100

        input_ids_list.append(input_ids)
        target_ids_list.append(target_ids)

    # Pad all sequences in the batch
    input_ids_padded = pad_sequence(input_ids_list, batch_first=True, padding_value=1)
    target_ids_padded = pad_sequence(target_ids_list, batch_first=True, padding_value=-100)

    return input_ids_padded, target_ids_padded

In [135]:
train_loader = DataLoader(train_set, batch_size=128, shuffle=True, collate_fn=pad_collate)

In [136]:
i, t = next(iter(train_loader))
i.shape, t.shape

(torch.Size([128, 493]), torch.Size([128, 493]))

In [138]:
i[0]

tensor([   1, 3687,  155, 4152, 1129,   15,  132,  269, 1129,  818, 5904, 2463,
        1665,   27, 4890,   13, 3681,   13, 2072,   15,  132,  269, 1129,  303,
         564,   27,  260,  161,  389, 1215,  155,  832, 2120,  241,  349, 3681,
         253,  455,  225, 1848,  281, 1215,    3,  161,  336, 1291,  161, 9036,
         759,  159, 5791, 4245,  132, 2371, 4170,  159, 1129,   27,  239,  161,
         389,  336,  375,   15,  216,  407,  162,  255, 1759,  241,  159, 3681,
          15,  195, 3681,  303,  302,  161,  910,  686, 1369,   15, 2343,  258,
         994, 1796,  241,  159, 3681,   15, 2343,  258,  442, 1463,  241,  159,
        3681,   15,  132,  132,  316,  252,   13,  260,  161,  389,  336, 2479,
          15,  216,  274,  162,  255,  155,  469, 1158,   15,  216,  548,  346,
         159, 3681,  161,  413,  155, 2120,   15,  195, 2120,  303,  832,  161,
         455,  225, 1848,  281, 1215, 1242,  260,  161,  389,  958,  440,  159,
        2120, 1056,   15,  132,  132,   

In [137]:
t[0]

tensor([-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,  161,  389,
         336,  375,   15,  216,  407,  162,  255, 1759,  241,  159, 3681,   15,
         195, 3681,  303,  302,  161,  910,  686, 1369,   15, 2343,  258,  994,
        1796,  241,  159, 3681,   15, 2343,  258,  442, 1463,  241,  159, 3681,
          15,  132,  132,  316,  252,   13,  260,  161,  389,  336, 2479,   15,
         216,  274,  162,  255,  155,  469, 1158,   15,  216,  548,  346,  159,
        3681,  161,  413,  155, 2120,   15,  195, 2120,  303,  832,  161,  455,
         225, 1848,  281, 1215, 1242,  260,  161,  389,  958,  440,  159, 2120,
        1056,   15,  132,  132,    3,   